<a href="https://colab.research.google.com/github/ArshnoorSinghh/ML-Project/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03"
df = pd.read_parquet(url, storage_options={"token": HF_TOKEN})

print(df.shape)
df.head()

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: a page is under-earning if its CTR is below the
typical CTR for its position. I rank pages by that gap, worst first, so
an editor reviews the biggest under-earners.

Reason code it outputs: LOW_CTR_FOR_POSITION
Action label: review_title_and_snippet

Signals my rule leans on (checked below):
1. CTR falls as search position gets worse (behind the CTR-fix flag).
2. Impression volume — does a page's visibility relate to its CTR gap.


Signal 1 verdict: CONFIRMED — mean CTR falls from top_3 (0.0048) down to
deep (0.0013) as position worsens; large n in every tier.
Signal 2 verdict: FALSE — mean CTR is nearly identical across low/medium/high
impression buckets (~0.003), so volume does not predict CTR. Good to know:
my rule should lean on position, not volume.

In [3]:
import pandas as pd
import numpy as np

# keep pages with search data, derive CTR and position
d = df[df["gsc_data_available"] == True].copy()
d["ctr"] = d["gsc_clicks"] / d["gsc_impressions"]
d["avg_position"] = d["gsc_sum_position"] / d["gsc_impressions"]

# position tiers (np.select from your sheet)
d["position_tier"] = np.select(
    [d["avg_position"] <= 3, d["avg_position"] <= 10, d["avg_position"] <= 20],
    ["top_3", "page_1", "page_2"], default="deep")

# SIGNAL 1: mean CTR per tier + n
print("Signal 1 — mean CTR by position tier:")
print(d.groupby("position_tier")["ctr"].mean())
print("\nn per tier:")
print(d.groupby("position_tier")["ctr"].count())

# SIGNAL 2: bucket by impression volume, mean CTR + n
d["imp_bucket"] = np.select(
    [d["gsc_impressions"] < 100, d["gsc_impressions"] < 1000],
    ["low", "medium"], default="high")

print("Signal 2 — mean CTR by impression bucket:")
print(d.groupby("imp_bucket")["ctr"].mean())
print("\nn per bucket:")
print(d.groupby("imp_bucket")["ctr"].count())

Signal 1 — mean CTR by position tier:
position_tier
deep      0.001289
page_1    0.003473
page_2    0.002770
top_3     0.004756
Name: ctr, dtype: float64

n per tier:
position_tier
deep       908354
page_1    1456122
page_2     519223
top_3      727362
Name: ctr, dtype: int64
Signal 2 — mean CTR by impression bucket:
imp_bucket
high      0.002714
low       0.003086
medium    0.003072
Name: ctr, dtype: float64

n per bucket:
imp_bucket
high        32419
low       2972453
medium     606189
Name: ctr, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rule: score each page by how far its CTR falls below the typical CTR for
its position (expected minus actual). Rank worst-first.
Reason code: LOW_CTR_FOR_POSITION. Action: review_title_and_snippet.


In [4]:
import numpy as np, os

# rebuild from scratch
d = df[df["gsc_data_available"] == True].copy()
d["ctr"] = d["gsc_clicks"] / d["gsc_impressions"]
d["avg_position"] = d["gsc_sum_position"] / d["gsc_impressions"]

# keep only pages shown enough to trust CTR
d = d[d["gsc_impressions"] >= 100].copy()

# position tiers
d["position_tier"] = np.select(
    [d["avg_position"] <= 3, d["avg_position"] <= 10, d["avg_position"] <= 20],
    ["top_3", "page_1", "page_2"], default="deep")

# expected CTR per tier, then the gap
d["expected_ctr"] = d.groupby("position_tier")["ctr"].transform("mean")
d["score"] = d["expected_ctr"] - d["ctr"]

d["reason_code"] = np.where(d["score"] > 0, "LOW_CTR_FOR_POSITION", "OK")
d["action"] = np.where(d["score"] > 0, "review_title_and_snippet", "no_action")

os.makedirs("work/outputs", exist_ok=True)
queue = d.sort_values("score", ascending=False)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("rows after filter:", len(d))
print("score range:", d["score"].min(), "to", d["score"].max())
queue[["content_hash_id","position_tier","gsc_impressions","ctr","expected_ctr","score","reason_code","action"]].head(10)

rows after filter: 638608
score range: -0.09861625919206266 to 0.003809202068988237


,content_hash_id,position_tier,gsc_impressions,ctr,expected_ctr,score,reason_code,action
9841360,content_bb56fccbc0d97668,top_3,245,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
4350159,content_a744e2af9f5fd079,top_3,774,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
4350172,content_9d9f35b48ca3c5e0,top_3,447,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
4350176,content_6f815c8049e6a744,top_3,415,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
4350222,content_0b6ef77844dbea7f,top_3,126,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
4350228,content_93ae6947bd4ec52d,top_3,131,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
4350230,content_596bd7bf346ccde9,top_3,374,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
9841099,content_1a0b77610cbbab34,top_3,104,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
9841106,content_35b0e7d419453b85,top_3,137,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet
517,content_3ea1da426a358e8f,top_3,361,0.0,0.003809,0.003809,LOW_CTR_FOR_POSITION,review_title_and_snippet


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-10 review: all ten are top_3 pages with 100+ impressions but 0 clicks,
so each is flagged LOW_CTR_FOR_POSITION -> review_title_and_snippet.
- Why flagged: they rank near the top yet earn no clicks, far below the
  average CTR for top_3 pages.
- What would make it wrong: a page may rank for queries no one actually
  clicks (wrong-intent or navigational), so 0 CTR could be normal, not a
  fixable title problem. Low click counts on a single month are also noisy.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: pages that rank for irrelevant queries will show 0 CTR but
aren't really fixable by a rewrite — my rule can't tell those apart yet.
Leakage check: my inputs are impressions, clicks, CTR, and position, all
knowable at decision time. I used no future-window or label-derived column
as an input. The score is derived from CTR, but CTR is an observed
present-time signal, not a future outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.